In [68]:
import pandas as pd
import numpy as np
import json
import logging
from typing import List, Tuple, Dict, Any, Optional
import warnings
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model
from datasets import Dataset

logger = logging.getLogger(__name__)
warnings.filterwarnings('ignore')

In [69]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [70]:
file_path = "/content/drive/MyDrive/Colab Notebooks/efsa_sentiment_classification.csv"
df = pd.read_csv(file_path)

In [71]:
def format_prompt_target(row):
    prompt = f"[INST] Analyze this financial text step by step:\n\nFinancial text: {row['Sentence']}\n\nPlease:\n1. Extract the company name\n2. Determine the GICS industry\n3. Classify the coarse event type\n4. Classify the fine event type\n5. Analyze the sentiment [/INST]\n\n"

    target = (
        f"Let me analyze this step by step:\n\n"
        f"1. Company: {row['Company']}\n"
        f"2. Industry: {row['Industry']}\n"
        f"3. Coarse Event: {row['Coarse-Grained Event']}\n"
        f"4. Fine Event: {row['Fine-Grained Event']}\n"
        f"5. Sentiment: {row['Sentiment']}"
    )
    return prompt, target

In [72]:
train = df[:822]
val = df[822:939]
test = df[939:]

In [73]:
def create_dataset(df):
  inputs = []
  labels = []

  for _, row in df.iterrows():
      prompt, target = format_prompt_target(row)
      inputs.append(prompt)
      labels.append(target)

  # Create HuggingFace dataset
  hf_dataset = Dataset.from_dict({"input_text": inputs, "target_text": labels})

  return hf_dataset

train_dataset = create_dataset(train)
val_dataset = create_dataset(val)
test_dataset = create_dataset(test)

In [74]:
from collections.abc import ValuesView
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    model_inputs = {"input_ids": [], "attention_mask": [], "labels": []}

    for inp, tgt in zip(examples["input_text"], examples["target_text"]):
        input_tokenized = tokenizer(inp, add_special_tokens=False)
        target_tokenized = tokenizer(tgt, add_special_tokens=False)

        full_input_ids = input_tokenized["input_ids"] + target_tokenized["input_ids"]

        if len(full_input_ids) > MAX_LENGTH:
            full_input_ids = full_input_ids[:MAX_LENGTH]

        attention_mask = [1] * len(full_input_ids)
        while len(full_input_ids) < MAX_LENGTH:
            full_input_ids.append(tokenizer.pad_token_id)
            attention_mask.append(0)

        labels = [-100] * len(input_tokenized["input_ids"]) + target_tokenized["input_ids"]

        while len(labels) < MAX_LENGTH:
            labels.append(-100)

        labels = labels[:MAX_LENGTH]

        model_inputs["input_ids"].append(full_input_ids)
        model_inputs["attention_mask"].append(attention_mask)
        model_inputs["labels"].append(labels)

    return model_inputs

train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)
test_tokenized = test_dataset.map(tokenize_function, batched=True, remove_columns=test_dataset.column_names)

Map:   0%|          | 0/822 [00:00<?, ? examples/s]

Map:   0%|          | 0/117 [00:00<?, ? examples/s]

Map:   0%|          | 0/234 [00:00<?, ? examples/s]

In [8]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 129.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 113.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninst

In [75]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [76]:
from peft import prepare_model_for_kbit_training

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", quantization_config=quantization_config)

model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [60]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

def compute_metrics(eval_preds):
    predictions, labels = eval_preds

    # Convert logits to token IDs
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    def extract_field(text, field_name):
        if not text or not isinstance(text, str):
            return "MISSING"

        text = text.replace("'\n", "").replace('"\n', "")

        for line in text.splitlines():
            line = line.strip()
            if field_name.lower() + ":" in line.lower():
                parts = line.split(":", 1)
                if len(parts) > 1:
                    extracted = parts[1].strip()
                    if field_name.lower() == "sentiment":
                        extracted = extracted.upper()[:3]
                        if extracted in ["POS", "NEG", "NEU"]:
                            return extracted

                    return extracted

        return "NOT_FOUND"

    company_preds, company_labels = [], []
    industry_preds, industry_labels = [], []
    coarse_preds, coarse_labels = [], []
    fine_preds, fine_labels = [], []
    sentiment_preds, sentiment_labels = [], []

    for pred, label in zip(decoded_preds, decoded_labels):
        # Extract each field
        company_preds.append(extract_field(pred, "Company"))
        company_labels.append(extract_field(label, "Company"))

        industry_preds.append(extract_field(pred, "Industry"))
        industry_labels.append(extract_field(label, "Industry"))

        coarse_preds.append(extract_field(pred, "Coarse Event"))
        coarse_labels.append(extract_field(label, "Coarse Event"))

        fine_preds.append(extract_field(pred, "Fine Event"))
        fine_labels.append(extract_field(label, "Fine Event"))

        sentiment_preds.append(extract_field(pred, "Sentiment"))
        sentiment_labels.append(extract_field(label, "Sentiment"))

    def calc_f1(y_true, y_pred, average='weighted'):
        # Handle cases where extraction failed
        valid_indices = [i for i in range(len(y_true))
                        if y_true[i] not in ["MISSING", "NOT_FOUND"] and
                           y_pred[i] not in ["MISSING", "NOT_FOUND"]]

        if len(valid_indices) == 0:
            return 0.0

        valid_true = [y_true[i] for i in valid_indices]
        valid_pred = [y_pred[i] for i in valid_indices]

        try:
            return f1_score(valid_true, valid_pred, average=average, zero_division=0)
        except:
            return 0.0

    def calc_accuracy(y_true, y_pred):
        valid_indices = [i for i in range(len(y_true))
                        if y_true[i] not in ["MISSING", "NOT_FOUND"] and
                           y_pred[i] not in ["MISSING", "NOT_FOUND"]]

        if len(valid_indices) == 0:
            return 0.0

        valid_true = [y_true[i] for i in valid_indices]
        valid_pred = [y_pred[i] for i in valid_indices]

        try:
            return accuracy_score(valid_true, valid_pred)
        except:
            return 0.0

    metrics = {
        'company_f1': calc_f1(company_labels, company_preds),
        'company_acc': calc_accuracy(company_labels, company_preds),

        'industry_f1': calc_f1(industry_labels, industry_preds),
        'industry_acc': calc_accuracy(industry_labels, industry_preds),

        'coarse_event_f1': calc_f1(coarse_labels, coarse_preds),
        'coarse_event_acc': calc_accuracy(coarse_labels, coarse_preds),

        'fine_event_f1': calc_f1(fine_labels, fine_preds),
        'fine_event_acc': calc_accuracy(fine_labels, fine_preds),

        'sentiment_f1': calc_f1(sentiment_labels, sentiment_preds),
        'sentiment_acc': calc_accuracy(sentiment_labels, sentiment_preds),
    }

    task_f1s = [metrics['company_f1'], metrics['industry_f1'],
                metrics['coarse_event_f1'], metrics['fine_event_f1'],
                metrics['sentiment_f1']]

    valid_f1s = [f1 for f1 in task_f1s if f1 > 0]

    if valid_f1s:
        metrics['macro_f1'] = np.mean(valid_f1s)
        metrics['weighted_f1'] = np.mean(valid_f1s)
    else:
        metrics['macro_f1'] = 0.0
        metrics['weighted_f1'] = 0.0

    return metrics

In [65]:
from transformers import TrainingArguments, EarlyStoppingCallback
import torch

torch.cuda.empty_cache()

training_args = TrainingArguments(
    output_dir="./lora_finetune_f1",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=1e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="weighted_f1",
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
    label_names=['labels']
)

data_collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt")

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.005
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping]
)

In [66]:
trainer.train()

Epoch,Training Loss,Validation Loss,Company F1,Company Acc,Industry F1,Industry Acc,Coarse Event F1,Coarse Event Acc,Fine Event F1,Fine Event Acc,Sentiment F1,Sentiment Acc,Macro F1,Weighted F1
1,No log,0.094838,0.678100,0.658120,0.770913,0.735043,0.681388,0.581197,0.743590,0.735043,0.947430,0.914530,0.764284,0.764284
2,No log,0.090718,0.686647,0.666667,0.775057,0.735043,0.710623,0.598291,0.801134,0.752137,0.947482,0.914530,0.784188,0.784188
3,0.059200,0.089706,0.682762,0.666667,0.794430,0.752137,0.712299,0.615385,0.827999,0.786325,0.947430,0.914530,0.792984,0.792984
4,0.059200,0.088768,0.674215,0.658120,0.800128,0.760684,0.722922,0.615385,0.852073,0.811966,0.947482,0.914530,0.799364,0.799364
5,0.041200,0.089952,0.674215,0.658120,0.808675,0.769231,0.758920,0.641026,0.834662,0.786325,0.952035,0.923077,0.805701,0.805701


TrainOutput(global_step=1030, training_loss=0.04984963287427587, metrics={'train_runtime': 2016.8049, 'train_samples_per_second': 2.038, 'train_steps_per_second': 0.511, 'total_flos': 8.986137074860032e+16, 'train_loss': 0.04984963287427587, 'epoch': 5.0})

In [67]:
model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/mistral_lora_model")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/mistral_lora_tokenizer")

('/content/drive/MyDrive/Colab Notebooks/mistral_lora_tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/mistral_lora_tokenizer/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/mistral_lora_tokenizer/chat_template.jinja',
 '/content/drive/MyDrive/Colab Notebooks/mistral_lora_tokenizer/tokenizer.model',
 '/content/drive/MyDrive/Colab Notebooks/mistral_lora_tokenizer/added_tokens.json',
 '/content/drive/MyDrive/Colab Notebooks/mistral_lora_tokenizer/tokenizer.json')

In [78]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast
from peft import PeftModel

print("Loading model and tokenizer...")
lora_base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map='auto', torch_dtype=torch.bfloat16, quantization_config=quantization_config)
lora_tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/Colab Notebooks/mistral_lora_tokenizer", use_fast=True)
lora_model = PeftModel.from_pretrained(lora_base_model, "/content/drive/MyDrive/Colab Notebooks/mistral_lora_model")

print('Model loaded successfully!')

Loading model and tokenizer...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded successfully!


In [79]:
lora_model.eval()

predictions = []
labels = []

with torch.no_grad():
    for i, example in enumerate(test_tokenized):
        input_ids = torch.tensor([example["input_ids"]]).to(lora_model.device)
        attention_mask = torch.tensor([example["attention_mask"]]).to(lora_model.device)
        label_ids = example["labels"]

        outputs = lora_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        predicted_token_ids = torch.argmax(logits, dim=-1).squeeze().cpu().numpy()

        predictions.append(predicted_token_ids)
        labels.append(label_ids)

predictions_array = np.array(predictions)
labels_array = np.array(labels)

print("Computing metrics...")
metrics = compute_metrics((predictions_array, labels_array))
print(metrics)

Computing metrics...
{'company_f1': 0.8587085028261499, 'company_acc': 0.8290598290598291, 'industry_f1': 0.8529065803198532, 'industry_acc': 0.811965811965812, 'coarse_event_f1': 0.8207203906329598, 'coarse_event_acc': 0.7393162393162394, 'fine_event_f1': 0.7177248767995701, 'fine_event_acc': 0.6837606837606838, 'sentiment_f1': 0.9691609204894228, 'sentiment_acc': 0.9401709401709402, 'macro_f1': np.float64(0.8438442542135911), 'weighted_f1': np.float64(0.8438442542135911)}


In [89]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
from collections import defaultdict

def compute_metrics_by_variant(eval_preds):

    labels, predictions = eval_preds

    # If logits, convert to token IDs
    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    # Replace ignore index -100 in labels with pad token id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    variants = {
        'EFSA': ['Company', 'Industry', 'Coarse Event', 'Fine Event', 'Sentiment'],
        'Coarse-grained EFSA': ['Company', 'Industry', 'Coarse Event', 'Sentiment'],
        'Fine-grained EFSA': ['Company', 'Industry', 'Fine Event', 'Sentiment'],
        'Entity-Level FSA': ['Company', 'Industry', 'Sentiment'],
    }

    def extract_field(text, field_name):
        if not text or not isinstance(text, str):
            return "MISSING"
        text = text.replace("'\n", "").replace('"\n', "")
        for line in text.splitlines():
            line = line.strip()
            if field_name.lower() + ":" in line.lower():
                parts = line.split(":", 1)
                if len(parts) > 1:
                    extracted = parts[1].strip()
                    if field_name.lower() == "sentiment":
                        extracted = extracted.upper()[:3]
                        if extracted in ["POS", "NEG", "NEU"]:
                            return extracted
                    return extracted
        return "NOT_FOUND"

    # Extract fields from all preds and labels once, store in dict of lists
    all_fields_preds = defaultdict(list)
    all_fields_labels = defaultdict(list)

    for pred_text, label_text in zip(decoded_preds, decoded_labels):
        for field in set(f for cols in variants.values() for f in cols):
            all_fields_preds[field].append(extract_field(pred_text, field))
            all_fields_labels[field].append(extract_field(label_text, field))

    def calc_f1(y_true, y_pred):
        valid_idx = [i for i in range(len(y_true))
                     if y_true[i] not in ["MISSING", "NOT_FOUND"] and y_pred[i] not in ["MISSING", "NOT_FOUND"]]
        if not valid_idx:
            return 0.0
        y_true_filtered = [y_true[i] for i in valid_idx]
        y_pred_filtered = [y_pred[i] for i in valid_idx]
        try:
            return f1_score(y_true_filtered, y_pred_filtered, average='weighted', zero_division=0)
        except:
            return 0.0

    def calc_accuracy(y_true, y_pred):
        valid_idx = [i for i in range(len(y_true))
                     if y_true[i] not in ["MISSING", "NOT_FOUND"] and y_pred[i] not in ["MISSING", "NOT_FOUND"]]
        if not valid_idx:
            return 0.0
        y_true_filtered = [y_true[i] for i in valid_idx]
        y_pred_filtered = [y_pred[i] for i in valid_idx]
        try:
            return accuracy_score(y_true_filtered, y_pred_filtered)
        except:
            return 0.0

    results = {}

    for variant, fields in variants.items():
        # Compose combined labels and preds by joining the field values with "|"
        y_true_combined = ["|".join(all_fields_labels[f][i] for f in fields) for i in range(len(decoded_labels))]
        y_pred_combined = ["|".join(all_fields_preds[f][i] for f in fields) for i in range(len(decoded_preds))]

        # Compute metrics
        acc = calc_accuracy(y_true_combined, y_pred_combined)
        f1_w = calc_f1(y_true_combined, y_pred_combined)

        results[variant] = {
            'accuracy': round(acc, 4),
            'f1_weighted': round(f1_w, 4),
        }

    return results

In [90]:
compute_metrics_by_variant((labels_array, predictions_array))

{'EFSA': {'accuracy': 0.3632, 'f1_weighted': 0.394},
 'Coarse-grained EFSA': {'accuracy': 0.4701, 'f1_weighted': 0.5107},
 'Fine-grained EFSA': {'accuracy': 0.4359, 'f1_weighted': 0.4647},
 'Entity-Level FSA': {'accuracy': 0.6538, 'f1_weighted': 0.6955}}